## Supporting notebook to prepare hydrodynamic data for notebook 0212

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import opentnsim.fis as fis
import xarray as xr
import networkx as nx
from shapely import transform
from shapely.geometry import Point
import geopandas as gpd
from opentnsim.environment.utils import (create_default_hydrodynamic_dataset, 
                                         add_specific_environmental_data_on_node, 
                                         interpolate_data_on_route,
                                         add_closest_node_to_xr_dataset)
from pyproj import CRS
%matplotlib inline

#### Loading raw data

In [2]:
path = os.getcwd()

In [3]:
df_wlev = pd.read_csv(os.path.join(path,"water_level_data_Terneuzen.csv"), delimiter=';', encoding='ISO-8859-1', on_bad_lines='skip', low_memory=False)

In [4]:
df_sal = pd.read_csv(os.path.join(path,"salinity_data_Terneuzen.csv"), delimiter=';', encoding='ISO-8859-1', on_bad_lines='skip', low_memory=False)
df_sal = df_sal[df_sal.GROOTHEID_OMSCHRIJVING == 'Saliniteit']

In [5]:
df_temp = pd.read_csv(os.path.join(path,"temperature_data_Terneuzen.csv"), delimiter=';', encoding='ISO-8859-1', on_bad_lines='skip', low_memory=False)

In [6]:
df = pd.concat([df_wlev, df_sal, df_temp])
df = df[df.REFERENTIEVLAK != 'WATSGL']
df = df.reset_index(drop=True)

#### Preparing time data

In [7]:
df['TIME'] = df['WAARNEMINGDATUM'] + ' ' + df['WAARNEMINGTIJD']
df['TIME'] = pd.to_datetime(df['TIME'], format = "%d-%m-%Y %H:%M:%S")

#### Selecting columns

In [8]:
df = df[['TIME','MEETPUNT_IDENTIFICATIE','NUMERIEKEWAARDE','GROOTHEID_OMSCHRIJVING','BEMONSTERINGSHOOGTE','LAT','LON','EPSG']]

#### Translating data

In [9]:
df = df.rename(columns={'MEETPUNT_IDENTIFICATIE':'Location',
                        'NUMERIEKEWAARDE':'Value',
                        'GROOTHEID_OMSCHRIJVING':'Physical quantity',
                        'BEMONSTERINGSHOOGTE':'Measurement height'})
df.loc[df['Physical quantity'] == 'Waterhoogte', 'Physical quantity'] = 'Water level'
df.loc[df['Physical quantity'] == 'Saliniteit', 'Physical quantity'] = 'Salinity'
df.loc[df['Physical quantity'] == 'Temperatuur', 'Physical quantity'] = 'Temperature'

#### Preparing data

In [10]:
#cm to m
df.loc[df['Physical quantity'] == 'Water level', 'Value'] = df.loc[df['Physical quantity'] == 'Water level', 'Value']/100
df['Measurement height'] = df['Measurement height']/100

In [11]:
df.loc[df[np.abs(df['Measurement height']) > 30].index, 'Measurement height'] = np.nan

df_wlev = df.loc[df['Physical quantity'] == 'Water level']
df.loc[df_wlev[np.abs(df_wlev['Value']) > 10].index, 'Value'] = np.nan

df_sal = df.loc[df['Physical quantity'] == 'Salinity']
df.loc[df_sal[(df_sal['Value'] > 35)|(df_sal['Value'] < 0)].index, 'Value'] = np.nan

df_temp = df.loc[df['Physical quantity'] == 'Temperature']
df.loc[df_temp[(df_temp['Value'] > 50)|(df_temp['Value'] < -10)].index, 'Value'] = np.nan

In [12]:
df = df.loc[df['Measurement height'].dropna().index]
df = df.sort_values('TIME')
df = df.reset_index(drop=True)

In [13]:
# Initialize a list to store cleaned indices
to_drop = []

for _, df_group in df.groupby(['Location','Physical quantity','Measurement height']):
    # Sort by TIME
    df_group = df_group.sort_values('TIME')

    # Identify duplicate TIME indices (keep the first)
    duplicated_times = df_group['TIME'][df_group['TIME'].duplicated(keep='first')]
    to_drop.extend(duplicated_times.index.tolist())

    # Remove duplicates for interpolation
    df_group_unique = df_group[~df_group['TIME'].duplicated(keep='first')]

    # Save mapping from TIME to original integer indexes
    time_to_idx = df_group_unique.index.to_series()

    # Set TIME as index for time-based interpolation
    df_group_unique = df_group_unique.set_index('TIME')

    # Interpolate
    interpolated = df_group_unique['Value'].interpolate(method='time').ffill().bfill()

    # Assign back using original integer indexes
    df.loc[time_to_idx, 'Value'] = interpolated.values

# Drop duplicates from the original df
df = df.drop(index=to_drop)
df = df.reset_index(drop=True)

In [14]:
df_sal = df[df['Physical quantity'] == 'Salinity']
df_station_avg_sal = (
    df_sal.groupby(['TIME', 'Location'], as_index=False)['Value']
      .mean()  # averages over all Measurement heights
)

metadata = df_sal.groupby('Location').agg({
    'Physical quantity':'first',
    'LAT': 'first',
    'LON': 'first',
    'EPSG': 'first'
}).reset_index()

# Merge back metaddata
df_station_avg_sal = df_station_avg_sal.merge(metadata, on='Location', how='left')
df_station_avg_sal = df_station_avg_sal.sort_values(['Location', 'TIME'])

In [15]:
df_without_sal = df[df['Physical quantity'] != 'Salinity']
df_without_sal = df_without_sal.drop(columns='Measurement height')

In [16]:
df_new = pd.concat([df_without_sal,df_station_avg_sal])
df_new = df_new.sort_values(['TIME','Location','Physical quantity'])
df_new = df_new.reset_index(drop=True)

In [17]:
df_new

,TIME,Location,Value,Physical quantity,LAT,LON,EPSG
0,2026-01-01 00:00:00,Sas van Gent,14.4320,Salinity,51.230267,3.807499,ETRS89
1,2026-01-01 00:00:00,Sas van Gent,2.0600,Water level,51.230267,3.807499,ETRS89
2,2026-01-01 00:00:00,Terneuzen,2.2700,Water level,51.336212,3.819812,ETRS89
3,2026-01-01 00:00:00,"Terneuzen, Sluiskilbrug",15.2915,Salinity,51.294000,3.835400,ETRS89
4,2026-01-01 00:00:00,"Terneuzen, Sluiskilbrug",7.0000,Temperature,51.294000,3.835400,ETRS89
...,...,...,...,...,...,...,...
66768,2026-03-01 23:50:00,"Terneuzen, westsluis, buiten",0.6600,Water level,51.337472,3.812564,ETRS89
66769,2026-03-02 00:00:00,Sas van Gent,2.1800,Water level,51.230267,3.807499,ETRS89
66770,2026-03-02 00:00:00,Terneuzen,0.7100,Water level,51.336212,3.819812,ETRS89
66771,2026-03-02 00:00:00,"Terneuzen, Sluiskilbrug",2.1800,Water level,51.294000,3.835400,ETRS89


#### Creating xr.DataSet

In [18]:
hydrodynamic_data = xr.Dataset()

In [19]:
df = df_new.copy()
# Ensure datetime
df['TIME'] = pd.to_datetime(df['TIME'], errors='coerce')
df = df[~df['TIME'].isna()]

# Define global time axis
t_min = df['TIME'].min()
t_max = df['TIME'].max()

dt = '10min'  # or '1H', '5min', etc.

common_time = pd.date_range(start=t_min, end=t_max, freq=dt)

In [20]:
df = df_new.copy()

# Global station list
all_stations = df['Location'].unique()

# Global time axis
t_min = df['TIME'].min()
t_max = df['TIME'].max()
common_time = pd.date_range(start=t_min, end=t_max, freq=dt)

hydrodynamic_data = xr.Dataset()

# --- Loop over quantities ---
for quantity, df_quantity in df.groupby('Physical quantity'):

    stations = []
    xs = []
    ys = []
    epsgs = []
    aligned_data = []

    for station in all_stations:

        df_location = df_quantity[df_quantity['Location'] == station]

        # --- Metadata (from full df) ---
        df_meta = df[df['Location'] == station]

        lat = df_meta['LAT'].mode().iloc[0] if 'LAT' in df_meta else np.nan
        lon = df_meta['LON'].mode().iloc[0] if 'LON' in df_meta else np.nan
        epsg_val = df_meta['EPSG'].mode().iloc[0] if 'EPSG' in df_meta else None

        try:
            crs = CRS.from_user_input(epsg_val)
            epsg_val = f"EPSG:{crs.to_epsg()}"
        except:
            pass

        stations.append(station)
        xs.append(lat)
        ys.append(lon)
        epsgs.append(epsg_val)

        # --- If no data for this station ---
        if df_location.empty:
            series = pd.Series(index=common_time, dtype=float, name=station)
            aligned_data.append(series)
            continue

        # --- Clean data ---
        df_loc = df_location.sort_values('TIME')

        # Remove duplicate timestamps
        df_loc = df_loc.drop_duplicates(subset='TIME')

        # Average over measurement height
        df_loc = df_loc.groupby('TIME')['Value'].mean()

        # --- Reindex to common time ---
        df_loc = df_loc.reindex(common_time)

        # --- Interpolate + fill ---
        df_loc = (
            df_loc
            .interpolate(method='time')
            .ffill()
            .bfill()
        )

        df_loc.name = station
        aligned_data.append(df_loc)

    # --- Combine stations ---
    df_aligned = pd.concat(aligned_data, axis=1)

    # --- Convert to xarray ---
    data = df_aligned.to_numpy().T
    time = common_time.to_numpy()

    da = xr.DataArray(
        data,
        dims=('STATION', 'TIME'),
        coords={
            'STATION': stations,
            'TIME': time,
            'LAT': ("STATION", xs),
            'LON': ("STATION", ys),
            'EPSG': ("STATION", epsgs)
        },
        name=quantity
    )

    hydrodynamic_data[quantity] = da

In [21]:
hydrodynamic_data["Salinity"].loc[dict(STATION="Terneuzen")] = hydrodynamic_data["Salinity"].loc[dict(STATION='Terneuzen, westsluis, buiten')]
hydrodynamic_data["Temperature"].loc[dict(STATION="Terneuzen")] = hydrodynamic_data["Temperature"].loc[dict(STATION='Terneuzen, westsluis, buiten')]
hydrodynamic_data["Temperature"].loc[dict(STATION='Sas van Gent')] = hydrodynamic_data["Temperature"].loc[dict(STATION='Terneuzen, Sluiskilbrug')]

In [23]:
hydrodynamic_data.sel({'TIME':pd.timerange(

<xarray.Dataset> Size: 899kB
Dimensions:      (STATION: 4, TIME: 8641)
Coordinates:
  * STATION      (STATION) <U28 448B 'Sas van Gent' ... 'Terneuzen, westsluis...
  * TIME         (TIME) datetime64[ns] 69kB 2026-01-01 ... 2026-03-02
    LAT          (STATION) float64 32B 51.23 51.34 51.29 51.34
    LON          (STATION) float64 32B 3.807 3.82 3.835 3.813
    EPSG         (STATION) <U9 144B 'EPSG:4258' 'EPSG:4258' ... 'EPSG:4258'
Data variables:
    Salinity     (STATION, TIME) float64 277kB 14.43 14.37 14.29 ... 22.36 22.36
    Temperature  (STATION, TIME) float64 277kB 7.0 7.3 7.1 7.0 ... 5.0 5.0 5.0
    Water level  (STATION, TIME) float64 277kB 2.06 2.05 2.05 ... 0.42 0.66 0.93

In [28]:
hydrodynamic_data = hydrodynamic_data.sel({'TIME':pd.date_range(pd.Timestamp('2026-01-01'),pd.Timestamp('2026-02-01'),freq=pd.Timedelta(minutes=10))})

In [30]:
hydrodynamic_data.to_netcdf('hydrodynamic_data_Terneuzen.nc')